# Earth–Sun distance illumination correction — **audit**

**Status.** The pipeline now removes the Earth–Sun distance term from L1B radiance **automatically**, at read time in the per-scene loaders, controlled by `radiometry.earth_sun_normalize` in `run_config.yaml` (default **on**). The single implementation is `emit_pics.earth_sun` (`spencer_E0` / `table_E0`), which this notebook imports — so the notebook can no longer disagree with the pipeline formula.

This notebook therefore no longer *applies* the correction as a one-off; it **audits** the built-in one:
1. computes per-scene `E0` and the pair flux ratio `E0(t2)/E0(t1)`,
2. tests whether that ratio correlates with Δt (the mechanism by which a seasonal `1/d²` term leaks into an apparent drift), and
3. reproduces the pipeline's E0 correction on the drift spectrum and **verifies equivalence**.

**Physics.** Top-of-atmosphere solar irradiance scales as `1/d²`, i.e. with the eccentricity factor `E0 = (d0/d)² = 1/d²` (`d` = Earth–Sun distance in AU) — about **±3.4%** across the year (perihelion ~0.983 AU early January, aphelion ~1.017 AU early July). At-sensor L1B radiance carries that factor; L2A reflectance always removes it. Normalizing radiance to 1 AU means **dividing by `E0`** = **multiplying by `d²`**. `E0` is a closed-form function of day-of-year — coefficients from **Iqbal, _An Introduction to Solar Radiation_ (1983), Eq. 1.2.1** (the accessible citation for the obscure primary note J. W. Spencer, _Search_ 2(5):172, 1971) — so no download is needed. An independent ephemeris table (`config/earth_sun_distance.txt`, tabulating `d` by day-of-year) is available as `radiometry.e0_method: table`; it agrees with the closed form to <0.1%.

**Companion:** `sza_leak_diagnostic.ipynb` (the solar-zenith illumination term).

> **No double-correction.** The loaded aggregated delta tables may *already* be E0-normalized (if produced with the toggle on). The correction cells below read the provenance sidecar's `earth_sun_normalize` flag and, when needed, reconstruct the **raw** radiance delta first — so the before/after contrast is always meaningful and the pipeline value is never corrected twice.

In [ ]:
# Repo-relative bootstrap: find the repo root (holds run_config.yaml) and import.
import sys, pathlib, json, glob
_p = pathlib.Path.cwd()
while not (_p / "run_config.yaml").exists() and _p != _p.parent:
    _p = _p.parent
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kendalltau, theilslopes

from emit_pics import paths
from emit_pics.run_config import load_config
from emit_pics.earth_sun import spencer_E0          # the single pipeline E0 implementation
from emit_pics.matched_pairs_drift_analysis import (
    load_aggregated_delta_tables, compute_drift_spectrum, DAYS_PER_YEAR)

cfg   = load_config()
ALPHA = cfg.alpha
SIZE  = cfg.patch_size

# Pool the aggregated (one-row-per-pair) delta tables across all PICS of this size.
df = load_aggregated_delta_tables(paths.DELTA_AGG_DIR, SIZE)
if df.empty:
    raise SystemExit(
        f"No aggregated delta tables for size {SIZE!r} in {paths.DELTA_AGG_DIR}.\n"
        "Run the analyze stage (scripts/run_PICS_allpatches_aggregate_pairs.py) first.")

n_pairs = df.groupby(["PICS_ID", "pair_id"]).ngroups
print(f"patch size {SIZE} km | {len(df)} rows | {df['PICS_ID'].nunique()} PICS | "
      f"{n_pairs} unique pairs | {df['band_index'].nunique()} channels")


def _delta_earth_sun_flag(agg_dir, size):
    """Read one delta sidecar to learn if the loaded radiance deltas already have
    E0 removed. Legacy tables (no flag in the sidecar) predate the toggle and are
    therefore raw."""
    for f in sorted(glob.glob(str(pathlib.Path(agg_dir) / f"*_{size}_delta.csv"))):
        sc = pathlib.Path(f + ".json")
        if sc.exists():
            rc = json.loads(sc.read_text()).get("run_config", {})
            if "earth_sun_normalize" in rc:
                return bool(rc["earth_sun_normalize"])
    return False

STORED_NORM = _delta_earth_sun_flag(paths.DELTA_AGG_DIR, SIZE)
print(f"config earth_sun_normalize = {cfg.earth_sun_normalize}   |   "
      f"loaded radiance deltas are "
      f"{'E0-NORMALIZED (pipeline)' if STORED_NORM else 'RAW (un-normalized)'}")

In [ ]:
# --- Adjustable plot y-limits -------------------------------------------------
# The drift-spectrum panels otherwise autoscale to the most extreme (often
# numerically unstable) channel, which flattens the ~0.5 %/yr differences between
# the series. Set tight limits here; use None to autoscale. Re-run the plot cells
# after changing these.
SPECTRUM_YLIM = (-1.5, 1.5)   # drift-spectrum panels, %/yr
DIAG_YLIM     = None          # signed-ΔSZA / flux-ratio scatter (degrees or %)

## 1. Per-scene eccentricity factor E0 (Spencer 1971)
`E0 = (d0/d)²` is the multiplicative factor on TOA solar irradiance (and hence on at-sensor radiance if not already normalised). `E0 > 1` near perihelion (Jan), `< 1` near aphelion (Jul).

In [ ]:
# Per-scene E0 via the shared pipeline implementation (emit_pics.earth_sun.spencer_E0),
# so the audit and the pipeline use the identical Spencer-1971 formula.
df = df.copy()
df["E0_t1"] = spencer_E0(df["t1_dt"])
df["E0_t2"] = spencer_E0(df["t2_dt"])

pairs = df.drop_duplicates(subset=["PICS_ID", "pair_id"]).copy()
pairs["flux_ratio_pct"] = 100.0 * (pairs["E0_t2"] / pairs["E0_t1"] - 1.0)
print(f"{len(pairs)} unique pairs")
print(f"E0 range across scenes : [{df['E0_t1'].min():.4f}, {df['E0_t1'].max():.4f}]  "
      f"(annual span ~ {100*(df['E0_t1'].max()/df['E0_t1'].min()-1):.1f}%)")
print(f"flux ratio E0(t2)/E0(t1) : mean {pairs['flux_ratio_pct'].mean():+.3f}%   "
      f"range [{pairs['flux_ratio_pct'].min():+.2f}, {pairs['flux_ratio_pct'].max():+.2f}]%")

## 2. Does the Earth–Sun flux ratio correlate with Δt?
A Δt-correlated flux ratio would map into an apparent radiance drift. Because the seasonal cycle is annual, correlation with Δt appears when the pair sampling is not seasonally balanced across baselines.

In [ ]:
m = pairs["flux_ratio_pct"].notna() & pairs["dt_days"].notna()
x = pairs.loc[m, "dt_days"].to_numpy()
y = pairs.loc[m, "flux_ratio_pct"].to_numpy()
tau, p = kendalltau(x, y)
slope, intc, lo, hi = theilslopes(y, x)
print(f"flux ratio (%) vs Δt :  Theil–Sen slope {slope * DAYS_PER_YEAR:+.4f} %/yr   "
      f"Kendall τ = {tau:+.3f}   p = {p:.2g}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].scatter(x, y, s=12, alpha=0.5)
xr = np.array([x.min(), x.max()])
ax[0].plot(xr, intc + slope * xr, "k-", lw=1.5)
ax[0].axhline(0.0, color="0.6", lw=0.8)
ax[0].set_xlabel("Δt (days)")
ax[0].set_ylabel("100·(E0(t2)/E0(t1) − 1)  [%]")
ax[0].set_title(f"Earth–Sun flux ratio vs Δt   (τ={tau:+.2f}, p={p:.2g})")
if DIAG_YLIM is not None:
    ax[0].set_ylim(DIAG_YLIM)
ax[1].hist(y, bins=40, color="tab:purple", alpha=0.85)
ax[1].axvline(0.0, color="k", lw=1.0)
ax[1].axvline(float(np.mean(y)), color="tab:red", lw=1.5, ls="--",
             label=f"mean {np.mean(y):+.2f}%")
ax[1].set_xlabel("flux ratio [%]")
ax[1].set_ylabel("pairs")
ax[1].set_title("distribution of Earth–Sun flux ratio")
ax[1].legend()
plt.tight_layout()
plt.show()

## 3. Reproduce & audit the pipeline's Earth–Sun (E0) correction
Reconstruct the **raw** radiance % change (undoing the pipeline's normalization if the loaded tables already carry it), then remove `E0(t2)/E0(t1)` exactly as the pipeline's per-scene loader does and refit. This is the matched-pairs, ratio-domain view of the same per-scene division the pipeline applies; when the loaded tables are already normalized, the manual result must reproduce the stored (pipeline) values — the equivalence check below confirms it. A large raw→corrected shift ⇒ the leak was real; a negligible one ⇒ EMIT L1B was effectively distance-normalized already (look to the SZA notebook).

In [ ]:
def remove_illumination(delta_pct, ratio):
    # Undo a multiplicative illumination factor  ratio = illum(t2) / illum(t1)
    # from a percent change. Measured (1 + dR/100) = ratio * (true ratio), so the
    # illumination-free percent change is 100 * ((1 + dR/100) / ratio - 1).
    delta_pct = np.asarray(delta_pct, dtype=float)
    ratio     = np.asarray(ratio, dtype=float)
    return 100.0 * ((1.0 + delta_pct / 100.0) / ratio - 1.0)


def apply_illumination(delta_pct, ratio):
    # Inverse of remove_illumination: re-impose the factor `ratio` on a percent
    # change. Used to reconstruct the RAW radiance delta from an already-E0-
    # normalized one (so the audit is never double-corrected).
    delta_pct = np.asarray(delta_pct, dtype=float)
    ratio     = np.asarray(ratio, dtype=float)
    return 100.0 * ((1.0 + delta_pct / 100.0) * ratio - 1.0)


def median_drift(spec):
    return float(np.nanmedian(spec["slope_pct_yr"]))


def plot_spectra(specs_labels_colors, title, ylim=None):
    # specs_labels_colors: list of (spectrum_df, label, color); ylim: (lo, hi) or None
    fig, ax = plt.subplots(figsize=(12, 5))
    for spec, lab, c in specs_labels_colors:
        s = spec.dropna(subset=["slope_pct_yr"]).sort_values("wavelength")
        ax.plot(s["wavelength"], s["slope_pct_yr"], ".", ms=5, color=c, label=lab)
    ax.axhline(0.0, color="k", lw=0.8)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.set_xlabel("wavelength (nm)")
    ax.set_ylabel("drift (%/yr)")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
f_es = (df["E0_t2"] / df["E0_t1"]).to_numpy()

# Reconstruct the RAW radiance delta whatever is stored, then apply the pipeline's
# E0 correction to it. If the loaded tables are already normalized, undo it first
# (apply_illumination) so nothing is double-corrected.
stored     = df["delta_rad_pct_median"].to_numpy()
delta_raw  = apply_illumination(stored, f_es) if STORED_NORM else stored
delta_corr = remove_illumination(delta_raw, f_es)     # == the pipeline's per-scene E0 division

df_raw  = df.copy(); df_raw["delta_rad_pct_median"]  = delta_raw
df_corr = df.copy(); df_corr["delta_rad_pct_median"] = delta_corr

spec_rad      = compute_drift_spectrum(df_raw,  "rad", alpha=ALPHA)   # before (raw)
spec_rad_corr = compute_drift_spectrum(df_corr, "rad", alpha=ALPHA)   # after  (E0-corrected)
spec_rfl      = compute_drift_spectrum(df,      "rfl", alpha=ALPHA)   # reflectance reference

print(f"median radiance drift  raw .............. {median_drift(spec_rad):+.3f} %/yr")
print(f"median radiance drift  E0-corrected ..... {median_drift(spec_rad_corr):+.3f} %/yr")
print(f"median reflectance drift (reference) .... {median_drift(spec_rfl):+.3f} %/yr")

# AUDIT: when the pipeline already corrected, the manual correction must reproduce it.
if STORED_NORM:
    dmax = float(np.nanmax(np.abs(delta_corr - stored)))
    print(f"\nAUDIT  max |manual − pipeline| radiance delta = {dmax:.2e} %  "
          f"-> {'PASS (equivalent)' if dmax < 1e-6 else 'CHECK — investigate'}")
else:
    print("\n(loaded tables are raw; 'E0-corrected' above is what the pipeline "
          "produces with radiometry.earth_sun_normalize: true)")

In [ ]:
plot_spectra(
    [(spec_rad,      "radiance — raw",                   "tab:red"),
     (spec_rad_corr, "radiance — E0-corrected (pipeline)", "tab:purple"),
     (spec_rfl,      "reflectance (reference)",          "tab:green")],
    "Radiance drift spectrum: raw vs Earth–Sun (E0) corrected — pipeline default",
    ylim=SPECTRUM_YLIM)

## Bonus: combined cos(SZA)·E0 illumination correction
The full first-order illumination factor is the **product** of the two ratios. This cell removes both at once — the closest matched-pairs analog to what the BRDF model does explicitly.

In [ ]:
# Combined cos(SZA)·E0, applied to the reconstructed RAW radiance delta (so this
# is raw -> full illumination-corrected, independent of what the tables stored).
f_both = ((np.cos(np.radians(df["SZA_t2"])) / np.cos(np.radians(df["SZA_t1"])))
          * (df["E0_t2"] / df["E0_t1"])).to_numpy()

df_both = df.copy()
df_both["delta_rad_pct_median"] = remove_illumination(delta_raw, f_both)
spec_rad_both = compute_drift_spectrum(df_both, "rad", alpha=ALPHA)

print(f"median radiance drift  cos(SZA)·E0-corrected .. {median_drift(spec_rad_both):+.3f} %/yr")
plot_spectra(
    [(spec_rad,      "radiance — raw",                   "tab:red"),
     (spec_rad_both, "radiance — cos(SZA)·E0-corrected", "tab:orange"),
     (spec_rfl,      "reflectance (reference)",          "tab:green")],
    "Radiance drift spectrum: raw vs full illumination-corrected",
    ylim=SPECTRUM_YLIM)

## How to read this
- **AUDIT PASS** (when the loaded tables are E0-normalized) confirms the pipeline's read-time per-scene normalization is numerically identical to this matched-pairs, ratio-domain correction — i.e. the built-in `radiometry.earth_sun_normalize` does exactly what this notebook derives.
- **raw → E0-corrected shift is large** ⇒ the seasonal `1/d²` term was a real leak in the *raw* radiance drift (exposed because the all-patches pool samples baselines with a seasonal imbalance); the pipeline's default-on correction removes it, pulling radiance toward the L2A reflectance reference.
- **shift is negligible** ⇒ EMIT L1B was effectively distance-normalized already; look to `sza_leak_diagnostic.ipynb` for the residual.
- The **combined `cos(SZA)·E0`** panel is the fullest first-order illumination correction available from matched pairs (E0 handled here + SZA handled explicitly); any residual radiance drift after it is the better candidate for genuine sensor behaviour — best confirmed with the pooled BRDF fit, which models `cosθ` illumination out directly.

**To see the raw vs corrected contrast explicitly**, regenerate the aggregated tables with `radiometry.earth_sun_normalize: false` (raw) — this notebook then reproduces, from those raw tables, exactly the corrected spectrum the pipeline produces with the toggle on.